# Validate Silver province reporting

## Purpose

Persist STOP and FLAG evidence for the D-57 reporting-area reference, boundaries, population, and project map before Gold runs. Evidence is written before the STOP gate.


## 1. Build deterministic context and checks


In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE validation_rule_version STRING
    DEFAULT 'silver-province-reporting-validation-v2';

CREATE OR REPLACE TEMPORARY VIEW silver_province_validation_context AS
SELECT
    (SELECT MAX(run_id) FROM `02-silver`.silver_province_reporting_area)
        AS area_run_id,
    (SELECT MAX(run_id) FROM `02-silver`.silver_province_reporting_area_boundary)
        AS boundary_run_id,
    (SELECT MAX(run_id)
     FROM `02-silver`.silver_province_reporting_area_population) AS population_run_id,
    (SELECT MAX(run_id) FROM `02-silver`.silver_project_province_map) AS map_run_id,
    (SELECT MAX(project_source_snapshot_id)
     FROM `02-silver`.silver_project_province_map) AS source_snapshot_id,
    SHA2(CONCAT_WS(
        '||',
        (SELECT MAX(run_id) FROM `02-silver`.silver_province_reporting_area),
        (SELECT MAX(run_id) FROM `02-silver`.silver_province_reporting_area_boundary),
        (SELECT MAX(run_id)
         FROM `02-silver`.silver_province_reporting_area_population),
        (SELECT MAX(run_id) FROM `02-silver`.silver_project_province_map),
        validation_rule_version
    ), 256) AS validation_run_id;

CREATE OR REPLACE TEMPORARY VIEW silver_province_region_population_compare AS
WITH area_by_region AS (
    -- the geographic reporting areas of each parent region. key 0 has no region
    SELECT
        parent_region_code,
        COALESCE(SUM(population_count), 0) AS area_population
    FROM `02-silver`.silver_province_reporting_area_population
    WHERE reporting_area_key <> 0
    GROUP BY parent_region_code
)

SELECT
    region.psgc_region_code,
    COALESCE(region.population_count, 0) AS region_population_count,
    COALESCE(area_by_region.area_population, 0) AS area_population
FROM `02-silver`.silver_region_population AS region
LEFT JOIN area_by_region
    ON region.psgc_region_code = area_by_region.parent_region_code
WHERE region.is_primary_source;

CREATE OR REPLACE TEMPORARY VIEW silver_province_validation_metrics AS
SELECT
    (SELECT COUNT(*) FROM `02-silver`.silver_project) AS source_projects,
    (SELECT COUNT(*) FROM `02-silver`.silver_project_province_map) AS mapped_rows,
    (SELECT COUNT(DISTINCT project_key)
     FROM `02-silver`.silver_project_province_map) AS mapped_projects,
    (SELECT COUNT(*) - COUNT(DISTINCT project_province_map_key)
     FROM `02-silver`.silver_project_province_map) AS duplicate_map_keys,
    (SELECT COUNT(*) - COUNT(DISTINCT reporting_area_key)
     FROM `02-silver`.silver_province_reporting_area) AS duplicate_area_keys,
    (SELECT COUNT_IF(reporting_area_key = 0)
     FROM `02-silver`.silver_province_reporting_area) AS unknown_members,
    (SELECT COUNT(*) - COUNT(DISTINCT reporting_area_key)
     FROM `02-silver`.silver_province_reporting_area_boundary)
        AS duplicate_boundary_grain,
    (SELECT COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
        'area', reporting_area_key, 'year', reference_year, 'source', source_name
     )) FROM `02-silver`.silver_province_reporting_area_population)
        AS duplicate_population_grain,
    (SELECT COUNT(*)
     FROM `02-silver`.silver_project_province_map AS map
     LEFT ANTI JOIN `02-silver`.silver_province_reporting_area AS area
        ON map.reporting_area_key = area.reporting_area_key) AS invalid_targets,
    (SELECT COUNT_IF(
        project_source_snapshot_id IS NULL OR project_run_id IS NULL
        OR project_ingest_run_id IS NULL OR psgc_run_id IS NULL
        OR boundary_run_id IS NULL OR mapping_config_version IS NULL
        OR mapping_rule_version IS NULL OR run_id IS NULL
     ) FROM `02-silver`.silver_project_province_map) AS broken_map_lineage,
    (SELECT COUNT_IF(coordinate_status <> 'VALID_PAIR')
     FROM `02-silver`.silver_project_province_map) AS missing_coordinates,
    (SELECT COUNT_IF(coordinate_status = 'VALID_PAIR' AND spatial_candidate_count = 0)
     FROM `02-silver`.silver_project_province_map) AS zero_spatial_candidates,
    (SELECT COUNT_IF(spatial_candidate_count > 1)
     FROM `02-silver`.silver_project_province_map) AS multiple_spatial_candidates,
    (SELECT COUNT_IF(conflict_flag)
     FROM `02-silver`.silver_project_province_map) AS conflicts,
    (SELECT COUNT_IF(reporting_area_key = 0)
     FROM `02-silver`.silver_project_province_map) AS unresolved_projects,
    (SELECT COUNT_IF(reporting_area_type = 'NCR')
     FROM `02-silver`.silver_project_province_map) AS ncr_projects,
    (SELECT COUNT_IF(reporting_area_type = 'INDEPENDENT_CITY')
     FROM `02-silver`.silver_project_province_map) AS independent_city_projects,
    (SELECT COUNT_IF(parent_region_code = '1900000000')
     FROM `02-silver`.silver_project_province_map) AS barmm_projects,
    (SELECT SUM(reported_budget_pesos) FROM `02-silver`.silver_project)
        AS source_budget,
    (SELECT SUM(project.reported_budget_pesos)
     FROM `02-silver`.silver_project_province_map AS map
     INNER JOIN `02-silver`.silver_project AS project
        ON map.project_key = project.project_key) AS mapped_budget,
    (SELECT SUM(population_count)
     FROM `02-silver`.silver_population_place_reconciliation
     WHERE row_disposition = 'ACCEPTED_PRIMARY'
       AND matched_place_type = 'BARANGAY' AND population_count > 0)
        AS accepted_population,
    (SELECT SUM(population_count)
     FROM `02-silver`.silver_province_reporting_area_population)
        AS reporting_area_population,
    -- d-57: each accepted barangay row may feed one reporting area once
    (SELECT SUM(source_row_count)
     FROM `02-silver`.silver_province_reporting_area_population)
        AS reporting_area_source_rows,
    (SELECT COUNT(DISTINCT NAMED_STRUCT(
        'file', source_file, 'sheet', sheet_name, 'row', source_row_number
     ))
     FROM `02-silver`.silver_population_place_reconciliation
     WHERE row_disposition = 'ACCEPTED_PRIMARY'
       AND matched_place_type = 'BARANGAY' AND population_count > 0)
        AS accepted_source_rows,
    -- every geographic reporting area belongs to exactly one official region
    (SELECT COUNT(*)
     FROM `02-silver`.silver_province_reporting_area AS area
     LEFT ANTI JOIN (
        SELECT DISTINCT psgc_code
        FROM `02-silver`.silver_psgc_place
        WHERE place_type = 'REGION'
     ) AS region
        ON area.parent_region_code = region.psgc_code
     WHERE area.reporting_area_key <> 0) AS areas_without_official_region,
    (SELECT COUNT(*) FROM `02-silver`.silver_province_reporting_area
     WHERE reporting_area_key <> 0) AS geographic_areas,
    -- region totals use the same accepted barangay rows, so the two levels must agree
    (SELECT SUM(population_count)
     FROM `02-silver`.silver_region_population
     WHERE is_primary_source) AS region_population,
    (SELECT COALESCE(SUM(population_count), 0)
     FROM `02-silver`.silver_province_reporting_area_population
     WHERE reporting_area_key = 0) AS unresolved_area_population,
    (SELECT MAX(excluded_population_count)
     FROM `02-silver`.silver_province_reporting_area_population
     WHERE reporting_area_key = 0) AS excluded_area_population,
    (SELECT COUNT_IF(area_population > region_population_count)
     FROM silver_province_region_population_compare) AS regions_over_population,
    (SELECT COUNT_IF(area_population < region_population_count)
     FROM silver_province_region_population_compare) AS regions_with_unresolved_population,
    (SELECT COUNT(*) FROM silver_province_region_population_compare) AS compared_regions,
    (SELECT SUM(region_population_count - area_population)
     FROM silver_province_region_population_compare) AS region_minus_area_population;

CREATE OR REPLACE TEMPORARY VIEW silver_province_validation_checks AS
WITH checks AS (
    SELECT 'silver_project_province_map' AS table_name, 'Uniqueness' AS attribute,
           'one final row per canonical project' AS check_name,
           ABS(source_projects - mapped_rows) + ABS(mapped_rows - mapped_projects)
               AS failed_rows, source_projects AS total_rows, 'stop' AS action
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Uniqueness', 'mapping keys are unique',
           duplicate_map_keys, mapped_rows, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Consistency', 'reported budget reconciles',
           CASE WHEN source_budget <=> mapped_budget THEN 0 ELSE 1 END, 1, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Validity', 'all targets resolve',
           invalid_targets, mapped_rows, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Lineage', 'mapping lineage is complete',
           broken_map_lineage, mapped_rows, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area', 'Uniqueness',
           'dimension keys and Unknown member are governed',
           duplicate_area_keys + ABS(unknown_members - 1),
           (SELECT COUNT(*) FROM `02-silver`.silver_province_reporting_area), 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_boundary', 'Uniqueness',
           'one boundary status per reporting area', duplicate_boundary_grain,
           (SELECT COUNT(*)
            FROM `02-silver`.silver_province_reporting_area_boundary), 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Uniqueness',
           'population grain is unique', duplicate_population_grain,
           (SELECT COUNT(*)
            FROM `02-silver`.silver_province_reporting_area_population), 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Consistency',
           'accepted-primary population reconciles',
           CASE WHEN accepted_population <=> reporting_area_population THEN 0 ELSE 1 END,
           1, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Uniqueness',
           'contributing Table C rows are counted once',
           CASE WHEN reporting_area_source_rows <=> accepted_source_rows THEN 0 ELSE 1 END,
           1, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area', 'Referential integrity',
           'every reporting area has one official parent region',
           areas_without_official_region, geographic_areas, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Consistency',
           'reporting-area population total equals region population total',
           CASE WHEN reporting_area_population <=> region_population THEN 0 ELSE 1 END,
           1, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Consistency',
           'no region has more reporting-area population than its own total',
           regions_over_population, compared_regions, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Consistency',
           'region minus reporting-area population equals unresolved population',
           CASE WHEN region_minus_area_population <=> unresolved_area_population THEN 0 ELSE 1 END,
           1, 'stop'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Completeness',
           'regions with accepted population on reporting area key 0',
           regions_with_unresolved_population, compared_regions, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Completeness',
           'people in accepted population that resolves to no reporting area',
           unresolved_area_population, reporting_area_population, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_province_reporting_area_population', 'Completeness',
           'people in Table C rows excluded before aggregation',
           COALESCE(excluded_area_population, 0),
           COALESCE(excluded_area_population, 0) + reporting_area_population, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Completeness', 'projects without usable coordinates',
           missing_coordinates, mapped_rows, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Completeness', 'valid points with zero candidates',
           zero_spatial_candidates, mapped_rows, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Uniqueness', 'points with multiple candidates',
           multiple_spatial_candidates, mapped_rows, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Consistency', 'name-coordinate conflicts',
           conflicts, mapped_rows, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Completeness', 'unresolved reporting areas',
           unresolved_projects, mapped_rows, 'flag'
    FROM silver_province_validation_metrics
    UNION ALL
    SELECT 'silver_project_province_map', 'Coverage',
           'NCR independent-city and BARMM coverage is reported',
           ncr_projects + independent_city_projects + barmm_projects, mapped_rows, 'flag'
    FROM silver_province_validation_metrics
)
SELECT
    context.validation_run_id,
    CASE checks.table_name
        WHEN 'silver_project_province_map' THEN context.map_run_id
        WHEN 'silver_province_reporting_area_population' THEN context.population_run_id
        WHEN 'silver_province_reporting_area_boundary' THEN context.boundary_run_id
        ELSE context.area_run_id
    END AS pipeline_run_id,
    'silver' AS layer, checks.table_name, context.source_snapshot_id,
    validation_rule_version AS rule_version, checks.attribute AS data_quality_attribute,
    checks.check_name AS data_quality_check, CAST(checks.failed_rows AS BIGINT) AS failed_rows,
    CAST(checks.total_rows AS BIGINT) AS total_rows,
    ROUND(100.0 * checks.failed_rows / NULLIF(checks.total_rows, 0), 4) AS percentage,
    CASE
        WHEN checks.action = 'stop' AND checks.failed_rows > 0 THEN 'FAIL'
        WHEN checks.action = 'flag' AND checks.failed_rows > 0 THEN 'FLAG'
        ELSE 'PASS'
    END AS status, checks.action,
    CONCAT('failed=', checks.failed_rows, '; total=', checks.total_rows) AS details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks
CROSS JOIN silver_province_validation_context AS context;


## 2. Persist evidence, then enforce STOP checks


In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL,
    layer STRING NOT NULL, table_name STRING NOT NULL, source_snapshot_id STRING,
    rule_version STRING, data_quality_attribute STRING NOT NULL,
    data_quality_check STRING NOT NULL, failed_rows BIGINT, total_rows BIGINT,
    percentage DOUBLE, status STRING NOT NULL, action STRING NOT NULL,
    details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING silver_province_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(COUNT_IF(status = 'FAIL' AND action = 'stop'),
           ' blocking Silver province-reporting checks failed')
)
FROM silver_province_validation_checks;


# Summary

The validator persists grain, key, target, lineage, project-count, budget, and population STOP checks before its gate. Coverage, ambiguity, conflicts, NCR, independent-city, and BARMM findings remain visible as FLAG evidence.
